# VRP Code-Generation Benchmark (Week 4)

Compare the same models as Week 3 (OpenRouter, OpenAI, Ollama, Gemini) on a **Nairobi grocery delivery** vehicle routing task:
- Generate synthetic orders, depots, vehicles (no hardcoded dataset).
- Ask the LLM to produce Python code that solves the VRP and writes `vrp_result.json`.
- Execute in sandbox, capture runtime, cost, feasibility, violations.
- Score: 50% cost, 20% feasibility, 15% runtime, 10% stability, 5% token efficiency.

In [ ]:
import os
from dotenv import load_dotenv
import gradio as gr

from vrp_benchmark import run_benchmark, get_models_for_provider, PROVIDERS
from route_plot import plot_routes
from data_generator import generate_dataset

load_dotenv(override=True)

## Run benchmark and build UI outputs

In [13]:
def run_vrp_benchmark(provider, model, n_orders, window_tightness, timeout_seconds):
    try:
        out = run_benchmark(
            provider=provider,
            model=model,
            n_orders=int(n_orders),
            window_tightness=window_tightness,
            seed=42,
            timeout_seconds=int(timeout_seconds),
        )
        code = out["code"]
        exec_result = out["execution"]
        log = ""
        if exec_result.get("stdout"):
            log += "STDOUT:\n" + exec_result["stdout"]
        if exec_result.get("stderr"):
            log += "\nSTDERR:\n" + exec_result["stderr"]
        if exec_result.get("error"):
            log += "\nError: " + str(exec_result["error"])
        cost_str = f"{out['total_cost']:.2f}" if out.get("total_cost") is not None else "N/A"
        violations_str = "\n".join(out["violations"]) if out["violations"] else "None"
        runtime_str = f"{out['runtime_seconds']:.2f}s" if out.get("runtime_seconds") is not None else "N/A"
        breakdown = out.get("breakdown", {})
        breakdown_str = " ".join(f"{k}: {v:.0f}" for k, v in breakdown.items())
        # Route plot
        dataset = generate_dataset(n_orders=int(n_orders), seed=42, window_tightness=window_tightness)
        result = exec_result.get("result")
        fig = plot_routes(dataset, result)
        return (
            code,
            log or "(no output)",
            cost_str,
            violations_str,
            runtime_str,
            f"{out['score']:.1f}/100",
            breakdown_str,
            out["summary"],
            fig,
        )
    except Exception as e:
        return (
            "",
            str(e),
            "N/A",
            "",
            "N/A",
            "0",
            "",
            f"Error: {e}",
            plot_routes(generate_dataset(n_orders=30, seed=42), None),
        )


def update_models(provider):
    models = get_models_for_provider(provider)
    return gr.update(choices=models, value=models[0] if models else None)

In [ ]:
with gr.Blocks(title="VRP Benchmark") as demo:
    gr.Markdown("## Nairobi Grocery Delivery VRP — Model Comparison")
    gr.Markdown("Same providers as Week 3. **OpenRouter** is used for API access (including OpenAI models)—set `OPENROUTER_API_KEY` in .env. Generates code that minimizes operational cost under time windows and capacity.")
    with gr.Row():
        provider = gr.Dropdown(choices=PROVIDERS, value="openrouter", label="Provider")
        models = get_models_for_provider("openrouter")
        model = gr.Dropdown(choices=models, value=models[0] if models else None, label="Model")
    with gr.Row():
        n_orders = gr.Slider(20, 200, value=30, step=5, label="Order count")
        window_tightness = gr.Dropdown(choices=["loose", "medium", "tight"], value="medium", label="Time window tightness")
        timeout_seconds = gr.Slider(30, 120, value=60, step=10, label="Execution timeout (s)")
    run_btn = gr.Button("Run benchmark")

    with gr.Row():
        code_out = gr.Textbox(label="Generated code", lines=12)
        log_out = gr.Textbox(label="Execution log", lines=8)
    with gr.Row():
        cost_out = gr.Textbox(label="Total cost")
        violations_out = gr.Textbox(label="Constraint violations", lines=4)
        runtime_out = gr.Textbox(label="Runtime")
    score_out = gr.Textbox(label="Score (0–100)")
    breakdown_out = gr.Textbox(label="Score breakdown (cost, feasibility, runtime, stability, tokens)")
    summary_out = gr.Textbox(label="Business summary", lines=3)
    plot_out = gr.Plot(label="Routes")

    provider.change(update_models, inputs=[provider], outputs=[model])
    run_btn.click(
        run_vrp_benchmark,
        inputs=[provider, model, n_orders, window_tightness, timeout_seconds],
        outputs=[code_out, log_out, cost_out, violations_out, runtime_out, score_out, breakdown_out, summary_out, plot_out],
    )

demo.launch()